Q1. Employee Performance: Population vs. Sample Variance
A company records the performance scores of eight employees:
`scores = [45, 50, 52, 55, 58, 60, 65, 75]`

(a) Write Python code to calculate the mean, population variance, sample variance, population standard deviation, and sample standard deviation. Implement the variance and standard deviation formulas manually without using `np.var()` or `np.std()` for the calculations.

In [32]:
import numpy as np
import pandas as pd

scores = [45, 50, 52, 55, 58, 60, 65, 75]
n = len(scores)
mean = sum(scores) / n
sq_dev = sum((x - mean) ** 2 for x in scores)

pop_var = sq_dev / n
samp_var = sq_dev / (n - 1)
pop_std = pop_var ** 0.5
samp_std = samp_var ** 0.5

print(f"Mean: {mean}")
print(f"Population Variance: {pop_var} | Population Std Dev: {pop_std}")
print(f"Sample Variance:     {samp_var} | Sample Std Dev:     {samp_std}")

Mean: 57.5
Population Variance: 77.25 | Population Std Dev: 8.789197915623474
Sample Variance:     88.28571428571429 | Sample Std Dev:     9.396047801374484


(b) Explain why the population and sample variance differ. Quantify the difference using the calculated results and recommend which variance formula should be used if these eight employees represent a sample of a much larger workforce.

In [33]:
df_denom = n - 1
diff_var = samp_var - pop_var
pct_diff = (diff_var / pop_var) * 100
ratio = samp_var / pop_var

print(f"Variance Difference: {diff_var} ({pct_diff}% higher) | Ratio: {ratio} (n/(n-1) = {n}/{df_denom})")
print(f"Reason: Sample variance divides by n-1 ({df_denom}) instead of N ({n}) to correct downward bias (Bessel's correction).")
print(f"Recommendation: Use Sample Variance ({samp_var}) and Sample Std Dev ({samp_std}) for a sample of a larger workforce.")

Variance Difference: 11.035714285714292 (14.285714285714294% higher) | Ratio: 1.142857142857143 (n/(n-1) = 8/7)
Reason: Sample variance divides by n-1 (7) instead of N (8) to correct downward bias (Bessel's correction).
Recommendation: Use Sample Variance (88.28571428571429) and Sample Std Dev (9.396047801374484) for a sample of a larger workforce.


Q2. Customer Spending: Comparing Data Spread
Use Seaborn's built-in `tips` dataset. Compare the `total_bill` and `tip` columns.

(a)Calculate the population variance and population standard deviation for both columns. Then calculate their means and coefficients of variation (CV). Use Python to compare their absolute and relative variability.

In [34]:
import seaborn as sns

tips = sns.load_dataset("tips")

def pop_stats(s):
    m = s.mean()
    v = ((s - m) ** 2).mean()
    sd = v ** 0.5
    cv = (sd / m) * 100
    return m, v, sd, cv

tb_m, tb_v, tb_sd, tb_cv = pop_stats(tips["total_bill"])
tp_m, tp_v, tp_sd, tp_cv = pop_stats(tips["tip"])

print(f"total_bill -> Mean: {tb_m}, Pop Var: {tb_v}, Pop Std: {tb_sd}, CV: {tb_cv}%")
print(f"tip        -> Mean: {tp_m}, Pop Var: {tp_v}, Pop Std: {tp_sd}, CV: {tp_cv}%")
print(f"Comparison: total_bill has higher absolute spread (Std {tb_sd} vs {tp_sd}), but tip has higher relative spread (CV {tp_cv}% vs {tb_cv}%).")

total_bill -> Mean: 19.78594262295082, Pop Var: 78.92813148851114, Pop Std: 8.884150577771132, CV: 44.90132589117038%
tip        -> Mean: 2.99827868852459, Pop Var: 1.9066085124966412, Pop Std: 1.3807999538298954, CV: 46.05308903120568%
Comparison: total_bill has higher absolute spread (Std 8.884150577771132 vs 1.3807999538298954), but tip has higher relative spread (CV 46.05308903120568% vs 44.90132589117038%).


(b) Identify the five largest `total_bill` values and calculate how the mean, variance, and standard deviation change when those observations are temporarily excluded. Explain why variance can be more sensitive to extreme observations than the mean.

In [35]:
top5 = tips["total_bill"].nlargest(5)
top5_list = top5.values.round(2).tolist()
filtered = tips["total_bill"].drop(top5.index)
f_m, f_v, f_sd, _ = pop_stats(filtered)

m_chg = (f_m - tb_m) / tb_m * 100
v_chg = (f_v - tb_v) / tb_v * 100
sd_chg = (f_sd - tb_sd) / tb_sd * 100

print(f"Top 5 total_bill: {top5_list}")
print(f"Mean:    {tb_m} -> {f_m} ({m_chg}%)")
print(f"Pop Var: {tb_v} -> {f_v} ({v_chg}%)")
print(f"Pop Std: {tb_sd} -> {f_sd} ({sd_chg}%)")
print("Reason: Variance squares deviations ((x - mean)^2), magnifying extreme values far more than the linear mean.")

Top 5 total_bill: [50.81, 48.33, 48.27, 48.17, 45.35]
Mean:    19.78594262295082 -> 19.191799163179915 (-3.0028564779204636%)
Pop Var: 78.92813148851114 -> 63.290054085187585 (-19.813059182326963%)
Pop Std: 8.884150577771132 -> 7.955504640510719 (-10.452838784429897%)
Reason: Variance squares deviations ((x - mean)^2), magnifying extreme values far more than the linear mean.


Q3. Build a Variance and Standard Deviation Calculator
Create a reusable Python program for the following datasets:
* `data_1 = [10, 20, 30, 40, 50]`
* `data_2 = [20, 20, 20, 20, 20]`
* `data_3 = [5, 10, 15, 20, 100]`
* `data_4 = [12, 14, 16, 18, 20, 22]`

(a) Implement separate functions to calculate the mean, population variance, sample variance, population standard deviation, and sample standard deviation without using NumPy's built-in statistical calculation functions. Correctly handle odd- and even-sized datasets.

In [36]:
def calc_mean(d):
    return sum(d) / len(d)

def calc_median(d):
    s, n = sorted(d), len(d)
    return float(s[n // 2]) if n % 2 else (s[n // 2 - 1] + s[n // 2]) / 2

def calc_pop_var(d):
    m = calc_mean(d)
    return sum((x - m) ** 2 for x in d) / len(d)

def calc_samp_var(d):
    m = calc_mean(d)
    return sum((x - m) ** 2 for x in d) / (len(d) - 1)

def calc_pop_std(d):
    return calc_pop_var(d) ** 0.5

def calc_samp_std(d):
    return calc_samp_var(d) ** 0.5

odd_data = [10, 20, 30, 40, 50]
even_data = [12, 14, 16, 18, 20, 22]

odd_mean, odd_med = calc_mean(odd_data), calc_median(odd_data)
even_mean, even_med = calc_mean(even_data), calc_median(even_data)

print(f"Odd (n=5)  -> Mean: {odd_mean}, Median: {odd_med}")
print(f"Even (n=6) -> Mean: {even_mean}, Median: {even_med}")

Odd (n=5)  -> Mean: 30.0, Median: 30.0
Even (n=6) -> Mean: 17.0, Median: 17.0


(b) Test all four datasets and produce a formatted report. Identify the dataset with zero variability and the dataset most affected by an extreme observation. Justify your conclusions using the calculated variance and standard deviation.

In [37]:
datasets = {
    "data_1": [10, 20, 30, 40, 50],
    "data_2": [20, 20, 20, 20, 20],
    "data_3": [5, 10, 15, 20, 100],
    "data_4": [12, 14, 16, 18, 20, 22]
}

print("Dataset  | n  |   Mean | Median |  Pop Var | Samp Var | Pop Std | Samp Std")
for name, d in datasets.items():
    cnt = len(d)
    m, med = calc_mean(d), calc_median(d)
    pv, sv = calc_pop_var(d), calc_samp_var(d)
    ps, ss = calc_pop_std(d), calc_samp_std(d)
    print(f"{name} | {cnt} | {m} | {med} | {pv} | {sv} | {ps} | {ss}")

d3_no_out = [5, 10, 15, 20]
d3_clean_var = calc_pop_var(d3_no_out)

print("Zero Variability: data_2 (Pop Var = 0.00, Std = 0.00) as all values equal the mean (20.00).")
print(f"Most Affected by Outlier: data_3 (Pop Var = 1250.00, Pop Std = 35.36); without 100, Pop Var is only {d3_clean_var}.")

Dataset  | n  |   Mean | Median |  Pop Var | Samp Var | Pop Std | Samp Std
data_1 | 5 | 30.0 | 30.0 | 200.0 | 250.0 | 14.142135623730951 | 15.811388300841896
data_2 | 5 | 20.0 | 20.0 | 0.0 | 0.0 | 0.0 | 0.0
data_3 | 5 | 30.0 | 15.0 | 1250.0 | 1562.5 | 35.35533905932738 | 39.528470752104745
data_4 | 6 | 17.0 | 17.0 | 11.666666666666666 | 14.0 | 3.415650255319866 | 3.7416573867739413
Zero Variability: data_2 (Pop Var = 0.00, Std = 0.00) as all values equal the mean (20.00).
Most Affected by Outlier: data_3 (Pop Var = 1250.00, Pop Std = 35.36); without 100, Pop Var is only 31.25.


Q4. Investigating the Effect of Mathematical Transformations
A researcher records these measurements:

(a)Create two new datasets: one by adding 10 to every observation and another by multiplying every observation by 4. Calculate the mean, population variance, and population standard deviation for all three datasets. Use Python to verify the mathematical relationships between the original and transformed statistics.

In [38]:
x = np.array([12, 15, 18, 20, 25, 30, 40], dtype=float)
add_10 = x + 10
mul_4 = x * 4

m_x, v_x, s_x = calc_mean(x), calc_pop_var(x), calc_pop_std(x)
m_a, v_a, s_a = calc_mean(add_10), calc_pop_var(add_10), calc_pop_std(add_10)
m_m, v_m, s_m = calc_mean(mul_4), calc_pop_var(mul_4), calc_pop_std(mul_4)

exp_m_a = m_x + 10
exp_m_m, exp_v_m, exp_s_m = 4 * m_x, 16 * v_x, 4 * s_x

print(f"Original (X) -> Mean: {m_x}, Pop Var: {v_x}, Pop Std: {s_x}")
print(f"Added (X+10) -> Mean: {m_a}, Pop Var: {v_a}, Pop Std: {s_a}")
print(f"Scaled (4X)  -> Mean: {m_m}, Pop Var: {v_m}, Pop Std: {s_m}")
print(f"Verify X+10: Mean = {m_a} (Expected: {exp_m_a}), Var = {v_a} (Expected: {v_x})")
print(f"Verify 4X:   Mean = {m_m} (Expected: {exp_m_m}), Var = {v_m} (Expected: {exp_v_m}), Std = {s_m} (Expected: {exp_s_m})")

Original (X) -> Mean: 22.857142857142858, Pop Var: 80.12244897959184, Pop Std: 8.951114398754596
Added (X+10) -> Mean: 32.857142857142854, Pop Var: 80.12244897959184, Pop Std: 8.951114398754596
Scaled (4X)  -> Mean: 91.42857142857143, Pop Var: 1281.9591836734694, Pop Std: 35.804457595018384
Verify X+10: Mean = 32.857142857142854 (Expected: 32.85714285714286), Var = 80.12244897959184 (Expected: 80.12244897959184)
Verify 4X:   Mean = 91.42857142857143 (Expected: 91.42857142857143), Var = 1281.9591836734694 (Expected: 1281.9591836734694), Std = 35.804457595018384 (Expected: 35.804457595018384)


(b) Create a fourth dataset by multiplying every observation by $-2$. Calculate its mean, variance, and standard deviation. Explain why a negative multiplier changes the direction of the values but does not make variance or standard deviation negative.

In [39]:
mul_neg2 = x * -2
neg_list = mul_neg2.tolist()
m_n, v_n, s_n = calc_mean(mul_neg2), calc_pop_var(mul_neg2), calc_pop_std(mul_neg2)

print(f"Scaled (-2X) -> Values: {neg_list}")
print(f"Mean: {m_n} (-2 * {m_x}) | Pop Var: {v_n} ((-2)^2 * {v_x}) | Pop Std: {s_n} (|-2| * {s_x})")

Scaled (-2X) -> Values: [-24.0, -30.0, -36.0, -40.0, -50.0, -60.0, -80.0]
Mean: -45.714285714285715 (-2 * 22.857142857142858) | Pop Var: 320.48979591836735 ((-2)^2 * 80.12244897959184) | Pop Std: 17.902228797509192 (|-2| * 8.951114398754596)


Q5. Outlier Detection and Robust Spread Analysis
A data analyst records daily sales, in thousands of rupees:
`sales = [12, 14, 15, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 120]`

(a) Write Python code to calculate the mean, median, population variance, and population standard deviation. Then identify potential outliers using the IQR method and report the quartiles, IQR, lower bound, upper bound, and flagged observations.

In [40]:
sales = [12, 14, 15, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 120]
s_mean, s_med = calc_mean(sales), calc_median(sales)
s_var, s_std = calc_pop_var(sales), calc_pop_std(sales)

q1, q3 = np.percentile(sales, [25, 75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = [v for v in sales if v < low or v > high]

print(f"Mean: {s_mean} | Median: {s_med} | Pop Var: {s_var} | Pop Std: {s_std}")
print(f"Q1: {q1} | Q3: {q3} | IQR: {iqr} | Bounds: [{low}, {high}] | Outliers: {outliers}")

Mean: 25.571428571428573 | Median: 18.5 | Pop Var: 699.3877551020408 | Pop Std: 26.445940238570472
Q1: 15.25 | Q3: 21.75 | IQR: 6.5 | Bounds: [5.5, 31.5] | Outliers: [120]


(b) Temporarily exclude the flagged outlier and recalculate the mean, median, variance, and standard deviation. Compare the percentage changes in variance and standard deviation, and recommend which statistics best describe typical daily sales while explaining why an extreme observation should be investigated before removal.

In [41]:
clean = [v for v in sales if v not in outliers]
c_mean, c_med = calc_mean(clean), calc_median(clean)
c_var, c_std = calc_pop_var(clean), calc_pop_std(clean)

m_pct = (c_mean - s_mean) / s_mean * 100
med_pct = (c_med - s_med) / s_med * 100
v_pct = (c_var - s_var) / s_var * 100
s_pct = (c_std - s_std) / s_std * 100

print(f"Mean:    {s_mean} -> {c_mean} ({m_pct}%)")
print(f"Median:  {s_med} -> {c_med} ({med_pct}%)")
print(f"Pop Var: {s_var} -> {c_var} ({v_pct}%)")
print(f"Pop Std: {s_std} -> {c_std} ({s_pct}%)")
print(f"Recommendation: Use Median ({s_med}k) and IQR ({iqr}k) when the outlier is present, as they resist extreme skew.")

Mean:    25.571428571428573 -> 18.307692307692307 (-28.40567253975076%)
Median:  18.5 -> 18.0 (-2.7027027027027026%)
Pop Var: 699.3877551020408 -> 14.520710059171599 (-97.92379692763522%)
Pop Std: 26.445940238570472 -> 3.810604946615642 (-85.59096438908983%)
Recommendation: Use Median (18.5k) and IQR (6.5k) when the outlier is present, as they resist extreme skew.
